In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Read Data from Silver Layer**

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source}")

In [0]:
display(df_silver)

### **Selecting Required Columns**

In [0]:
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")

In [0]:
display(df_gold.limit(5))

### **Write the Final Data Frame to Gold Schema**

In [0]:
df_gold.write \
    .format("delta") \
        .option("delta.enableChangeDataFeed", True) \
            .mode("overwrite") \
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dim{data_source}")

### **Merging Child Company Data with Parent Company**

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")

df_child_products = spark.sql(f" SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dimproducts;")
df_child_products.show(5)

In [0]:
delta_table.alias("target").merge(
    source = df_child_products.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "division" : "source.division",
        "category" : "source.category",
        "product" : "source.product",
        "variant" : "source.variant"
    }
).whenNotMatchedInsert(
    values = {
        "product_code" : "source.product_code",
        "division" : "source.division",
        "category" : "source.category",
        "product" : "source.product",
        "variant" : "source.variant"
    }
).execute()